# Denetimsiz Öğrenme: K-Means Kümeleme ve Temel Bileşen Analizi (PCA)

Bu modül; hedef etiketlerin (y) bulunmadığı durumlarda veri noktalarındaki gizli desenleri, doğal grupları keşfeden **K-Means Kümeleme Algoritması**'nı ve yüksek boyutlu verileri en yüksek varyansı koruyarak 2B/3B düzleme indirgeyen **Temel Bileşen Analizi (Principal Component Analysis - PCA)** yöntemini inceler.

---

## 1. Matematiksel Temel

### K-Means Algoritması
Veri noktalarını $k$ adet kümeye ayırmak için Küme İçi Kareler Toplamını (Within-Cluster Sum of Squares - WCSS) minimize eder:
$$\text{WCSS} = \sum_{i=1}^k \sum_{x \in S_i} \|x - \mu_i\|^2$$
1. **Başlatma (K-Means++):** İlk merkezler birbirinden olabildiğince uzak seçilerek yerel minimumlara takılma riski azaltılır.
2. **Atama:** Her $x$ noktası en yakın ağırlık merkezine ($\mu_i$) atanır.
3. **Güncelleme:** Ağırlık merkezleri, o kümeye atanan noktaların ortalaması olarak güncellenir:
   $$\mu_i = \frac{1}{|S_i|} \sum_{x \in S_i} x$$

### Temel Bileşen Analizi (PCA)
Öznitelikler arasındaki doğrusal korelasyonu ortadan kaldırarak verinin kovaryans matrisinin $\Sigma$ özdeğer (eigenvalue $\lambda$) ve özvektörlerini (eigenvector $v$) bulur:
$$\Sigma v = \lambda v$$
En büyük özdeğere sahip ilk $m$ adet özvektör, maksimum varyans yönlerini temsil eden temel bileşenleri (Principal Components) oluşturur.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

# 1. Sentetik Çok Boyutlu Küme Verisi Oluşturma (6 Öznitelik, 4 Küme)
X, y_true = make_blobs(n_samples=500, n_features=6, centers=4, cluster_std=1.2, random_state=42)

# Boyut indirgeme ve kümeleme öncesi standartlaştırma
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"Veri Matrisi Şekli: {X_scaled.shape}")


## 2. Optimum Küme Sayısı $k$'nın Belirlenmesi: Dirsek (Elbow) Yöntemi ve Siluet Skoru

In [ ]:
wcss = []
silhouette_scores = []
K_range = range(2, 9)

for k in K_range:
    kmeans = KMeans(n_clusters=k, init='k-means++', random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    wcss.append(kmeans.inertia_)
    silhouette_scores.append(silhouette_score(X_scaled, kmeans.labels_))

fig, ax1 = plt.subplots(figsize=(10, 5))

# WCSS grafiği
color = 'tab:blue'
ax1.set_xlabel('Küme Sayısı (k)')
ax1.set_ylabel('WCSS (Inertia)', color=color)
ax1.plot(K_range, wcss, 'bo-', linewidth=2)
ax1.tick_params(axis='y', labelcolor=color)

# Siluet Skoru grafiği
ax2 = ax1.twinx()
color = 'tab:red'
ax2.set_ylabel('Siluet Skoru', color=color)
ax2.plot(K_range, silhouette_scores, 'ro--', linewidth=2)
ax2.tick_params(axis='y', labelcolor=color)

plt.title("Optimum k Belirleme: Dirsek Yöntemi ve Siluet Skoru Analizi")
plt.grid(True, alpha=0.3)
plt.show()


## 3. PCA ile 2 Boyuta İndirgeme ve Küme Görselleştirmesi

In [ ]:
# Optimum k=4 ile K-Means modelini eğitme
optimal_k = 4
kmeans = KMeans(n_clusters=optimal_k, init='k-means++', random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(X_scaled)

# 6 boyuttan 2 boyuta PCA projeksiyonu
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

explained_variance = pca.explained_variance_ratio_
print(f"İlk 2 Bileşenin Açıkladığı Varyans: %{sum(explained_variance) * 100:.2f} (PC1: %{explained_variance[0]*100:.2f}, PC2: %{explained_variance[1]*100:.2f})")

plt.figure(figsize=(10, 7))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=cluster_labels, cmap='viridis', alpha=0.7, edgecolors='k')
plt.title(f"PCA 2B Düzleminde K-Means Kümeleri (Açıklanan Varyans: %{sum(explained_variance)*100:.1f})")
plt.xlabel("1. Temel Bileşen (PC1)")
plt.ylabel("2. Temel Bileşen (PC2)")
plt.colorbar(scatter, label='Küme No')
plt.grid(True, alpha=0.3)
plt.show()


## 4. Mühendislik Çıkarımları

1. **Ölçekleme Zorunluluğu:** K-Means ve PCA Öklid mesafelerine dayanır. Ölçeklendirme yapılmazsa büyük varyansa sahip öznitelikler analizi domine eder.
2. **Elbow vs Silhouette:** Dirsek yöntemi keskin bir kırılma göstermediğinde, -1 ile +1 arasında değer alan Siluet Skoru en net küme ayrışımını verir.
3. **PCA Sınırları:** PCA yalnızca doğrusal varyansı yakalar; doğrusal olmayan manifoldlar için t-SNE veya UMAP tercih edilir.
